In [0]:
dbutils.widgets.text("business_date", "", "Business Date (YYYY-MM-DD, empty=latest)")

import sys
sys.path.append("/Workspace/Shared")
import aws_utils

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit

spark = SparkSession.builder.getOrCreate()

env = aws_utils.get_env_prefix()

business_date_param = dbutils.widgets.get("business_date").strip()

if business_date_param:
    run_date = business_date_param
else:
    run_date = str(
        spark.table(f"`{env}-cdh`.primary.exposure")
        .filter(col("SourceSystem") == "profits")
        .agg({"BusinessDate": "max"})
        .collect()[0][0]
    )

run_date_lit = lit(run_date).cast("date")

In [0]:
# ==================== Subsidy Perimeter ====================
subsidy_base = (
    spark.table(f"`{env}-cdh`.landed.rpt_applicationsubsidy")
    .filter(
        (col("CurrentRecord") == 1)
        & col("platformName").isin("Bridge1", "Bridge2", "L4605")
    )
)

subsidy_new_accounts = (
    subsidy_base
    .select(col("new_account_Number").alias("AccountKey"))
)


subsidy_l2_accounts = (
    subsidy_base
    .filter(col("l2_account").isNotNull())
    .select(col("l2_account").alias("AccountKey"))
)

subsidy_perimeter = (
    subsidy_new_accounts
    .unionByName(subsidy_l2_accounts)
    .distinct()
)

print(f"Subsidy perimeter accounts: {subsidy_perimeter.count():,}")

In [0]:
# ==================== Collections (Matched Payments) ====================

_max_portfolio_date = (
    spark.table(f"`{env}-cdh`.primary.portfolio")
    .agg(F.max("BusinessDate")).collect()[0][0]
)
portfolio = (
    spark.table(f"`{env}-cdh`.primary.portfolio")
    .filter(col("BusinessDate") == _max_portfolio_date)
    .select("PortfolioId", "PortfolioName")
)

all_exposures = (
    spark.sql(f"SELECT DISTINCT ExposureKey, PortfolioName FROM `{env}-cdh`.primary.get_all_exposures('{run_date}')")
    .join(F.broadcast(portfolio), on="PortfolioName", how="inner")
    .select(col("ExposureKey"), col("PortfolioId"))
    .distinct()
)

exposure_in_perimeter = (
    all_exposures
    .withColumn("CollectionExposureKey",
        F.when(col("ExposureKey").rlike("^[A-Za-z]{3}[0-9]"),
               F.expr("substring(ExposureKey, 4)"))   
         .otherwise(col("ExposureKey"))                
    )
    .join(
        F.broadcast(subsidy_perimeter),
        (col("ExposureKey") == col("AccountKey")) |
        (col("CollectionExposureKey") == col("AccountKey")),
        "inner"
    )
    .select(
        col("ExposureKey").alias("AssetKey"),
        col("CollectionExposureKey"),
        col("PortfolioId"),
    )
    .distinct()
)

max_coll_date = (
    spark.table(f"`{env}-cdh`.primary.collection")
    .agg(F.max("BusinessDate"))
    .collect()[0][0]
)

collections = (
    spark.table(f"`{env}-cdh`.primary.collection")
    .filter(
        (col("BusinessDate") == max_coll_date) &
        (F.lower(col("FinalReason")) == "matched")
    )
)

accountpayment_df = (
    collections
    .join(
        exposure_in_perimeter,
        collections["ExposureKey"] == exposure_in_perimeter["CollectionExposureKey"],  
        "inner"
    )
    .select(
        col("PortfolioId"),
        col("AssetKey").alias("AccountNumber"),
        col("Amount"),
        col("ActualDate").cast("date").alias("ActualDate"),
        run_date_lit.alias("datadate"),
    )
)

print(f"dtransfer_accountpayment rows: {accountpayment_df.count():,}")

In [0]:
# ==================== Write dtransfer_accountpayment ====================

target_accountpayment = f"`{env}-cdh-delivery`.it.dtransfer_accountpayment"

(
    accountpayment_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_accountpayment)
)

print(f"Written to {target_accountpayment}")